### Step 1 — Configure project root and Python path

Purpose: Locate the repository root by walking up from the current working directory, then add `src/` to `sys.path` so `ibnr_utils` modules are importable without installation.  
Uses: `pathlib.Path`, `sys.path`.  
Produces: `project_root` — a `Path` object pointing to the repository root.

In [1]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "README.md").exists() and (path / "src" / "ibnr_utils").exists():
            return path
    raise FileNotFoundError("Could not find IBNR_Utils project root.")


project_root = find_project_root(Path.cwd())
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

print("Project root:", project_root)

Project root: /mnt/data/Linux/Documents/IBNR_Utils


### Step 2 — Import ibnr_utils functions

Purpose: Import the four functions used in this notebook.  
Uses: `aggregate_database_calendar_aligned`, `database_to_triangle_collection`, `read_triangle_database_csv`, `validate_all`.  
Produces: Module symbols available in the session.

In [2]:
from ibnr_utils import (
    aggregate_database_calendar_aligned,
    database_to_triangle_collection,
    read_triangle_database_csv,
    validate_all,
)

### Step 3 — Load the monthly triangle database

Purpose: Build the source path and read the long-format monthly CSV into a DataFrame.  
Uses: `read_triangle_database_csv`.  
Produces: `monthly_database` — the standard long-format DataFrame used as the source for all aggregations in this notebook.

In [3]:
database_path = project_root / "data" / "input" / "generated_monthly_triangles_database.csv"
monthly_database = read_triangle_database_csv(database_path)
monthly_database.shape

(65340, 9)

### Step 4 — Aggregate from monthly to quarterly periods

Purpose: Collapse monthly rows into calendar-aligned quarterly periods using the correct cumulative/incremental aggregation logic.  
Uses: `aggregate_database_calendar_aligned(source_basis="month", target_basis="quarter")`.  
Produces: `quarterly_database` — a long-format DataFrame with `basis="quarter"`.  

Interpretation: For cumulative amounts, aggregation keeps the latest source calendar cell inside each target period before summing; for incremental amounts it sums source cells directly. This ensures cumulative triangles remain consistent when collapsed to a coarser period.

In [4]:
quarterly_database = aggregate_database_calendar_aligned(
    monthly_database,
    source_basis="month",
    target_basis="quarter",
)

quarterly_database.shape

(7380, 9)

### Step 5 — Convert quarterly database to triangle collection and inspect

Purpose: Pivot the quarterly database into one upper-triangular DataFrame per concept, then display the Loss Incurred triangle to verify dimensions and `NaN` pattern.  
Uses: `database_to_triangle_collection`.  
Produces: `quarterly_triangles` — a dict mapping concept name to quarterly triangle DataFrame.

In [5]:
quarterly_triangles = database_to_triangle_collection(quarterly_database)
quarterly_triangles["Loss Incurred"]

,dev_0,dev_1,dev_2,dev_3,dev_4,dev_5,dev_6,dev_7,dev_8,dev_9,...,dev_30,dev_31,dev_32,dev_33,dev_34,dev_35,dev_36,dev_37,dev_38,dev_39
accident_period,,,,,,,,,,,,,,,,,,,,,
2016Q1,5.248538e+05,5.592843e+05,5.953964e+05,6.132237e+05,6.426764e+05,6.669179e+05,6.216181e+05,6.842418e+05,6.797318e+05,6.426592e+05,...,548884.404737,541365.992328,544835.378795,552074.496831,544100.903940,550526.214876,549713.812550,533985.152991,545664.497044,543642.946284
2016Q2,6.713612e+05,6.801421e+05,7.333755e+05,6.728997e+05,7.985850e+05,7.233171e+05,7.446545e+05,7.800045e+05,7.846065e+05,7.530155e+05,...,644848.194518,628289.248052,633315.504194,632940.255966,622932.351942,636840.615304,641984.687017,628623.738341,624834.395999,NaN
2016Q3,7.867210e+05,7.290972e+05,7.373736e+05,8.173847e+05,8.383125e+05,8.441777e+05,8.738999e+05,8.755927e+05,8.941187e+05,8.737558e+05,...,714238.762535,712182.726011,712266.896254,713232.113570,712223.373088,707319.331199,707722.221658,708142.379652,NaN,NaN
2016Q4,6.723014e+05,7.395269e+05,7.470856e+05,7.673591e+05,6.802908e+05,7.396248e+05,8.084522e+05,7.743650e+05,7.742824e+05,7.693679e+05,...,637140.833990,650137.925170,640856.008132,645661.074607,639793.689051,639179.882623,640263.535411,NaN,NaN,NaN
2017Q1,5.856406e+05,6.782561e+05,7.019179e+05,7.173522e+05,7.571934e+05,7.210626e+05,7.976371e+05,7.486266e+05,7.282303e+05,7.445818e+05,...,605325.121395,600205.123120,602735.691752,612613.400198,608669.761121,607552.731596,NaN,NaN,NaN,NaN
2017Q2,6.324114e+05,7.061322e+05,7.385658e+05,7.608590e+05,7.873632e+05,8.638859e+05,8.419951e+05,8.526103e+05,7.969978e+05,8.006993e+05,...,704513.057709,689451.801951,690640.461829,686784.756744,693684.083292,NaN,NaN,NaN,NaN,NaN
2017Q3,8.722715e+05,8.701235e+05,9.036511e+05,8.212093e+05,8.944481e+05,1.045556e+06,9.539421e+05,9.298059e+05,9.256097e+05,9.173256e+05,...,766604.631022,781517.989543,771989.790863,762698.250368,NaN,NaN,NaN,NaN,NaN,NaN
2017Q4,6.598299e+05,7.616010e+05,7.889466e+05,7.722989e+05,8.485646e+05,8.442165e+05,8.146821e+05,8.090694e+05,8.392996e+05,8.188436e+05,...,677111.057370,672842.513511,683717.168043,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018Q1,7.288786e+05,7.467464e+05,7.687827e+05,7.004141e+05,7.203281e+05,8.254315e+05,8.039749e+05,7.804673e+05,7.639152e+05,7.496638e+05,...,651850.308918,648368.331112,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Step 6 — Validate quarterly triangle relationships

Purpose: Run all built-in cross-concept checks on the quarterly triangle collection.  
Uses: `validate_all`.  
Produces: A printed pass/fail message confirming the quarterly triangles are internally consistent.

In [6]:
validate_all(quarterly_triangles)

All validations passed.


### Step 7 — Aggregate from monthly to yearly periods

Purpose: Collapse monthly rows into calendar-aligned yearly periods.  
Uses: `aggregate_database_calendar_aligned(source_basis="month", target_basis="year")`.  
Produces: `yearly_database` — a long-format DataFrame with `basis="year"`.

In [7]:
yearly_database = aggregate_database_calendar_aligned(
    monthly_database,
    source_basis="month",
    target_basis="year",
)

yearly_database.shape

(495, 9)

### Step 8 — Convert yearly database to triangle collection and inspect

Purpose: Pivot the yearly database into one upper-triangular DataFrame per concept, then display the Loss Incurred triangle.  
Uses: `database_to_triangle_collection`.  
Produces: `yearly_triangles` — a dict mapping concept name to yearly triangle DataFrame.

In [8]:
yearly_triangles = database_to_triangle_collection(yearly_database)
yearly_triangles["Loss Incurred"]

,dev_0,dev_1,dev_2,dev_3,dev_4,dev_5,dev_6,dev_7,dev_8,dev_9
accident_period,,,,,,,,,,
2016,2.747998e+06,2.953365e+06,3.027204e+06,2.813659e+06,2.627474e+06,2.539883e+06,2.519620e+06,2.535465e+06,2.527547e+06,2.516883e+06
2017,2.985871e+06,3.484742e+06,3.261486e+06,3.008256e+06,2.830045e+06,2.763455e+06,2.753714e+06,2.752806e+06,2.747652e+06,NaN
2018,3.134833e+06,3.612844e+06,3.516405e+06,3.215038e+06,3.006993e+06,2.894123e+06,2.920185e+06,2.894836e+06,NaN,NaN
2019,3.192526e+06,3.625429e+06,3.625979e+06,3.432152e+06,3.200415e+06,3.034070e+06,3.043825e+06,NaN,NaN,NaN
2020,3.463149e+06,4.095522e+06,3.923961e+06,3.705934e+06,3.446844e+06,3.296026e+06,NaN,NaN,NaN,NaN
2021,4.175165e+06,4.521168e+06,4.338064e+06,4.087881e+06,3.879976e+06,NaN,NaN,NaN,NaN,NaN
2022,4.199691e+06,4.988215e+06,4.690895e+06,4.428242e+06,NaN,NaN,NaN,NaN,NaN,NaN
2023,4.358153e+06,5.232024e+06,4.876668e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024,4.643697e+06,5.369977e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Step 9 — Validate yearly triangle relationships

Purpose: Run all built-in cross-concept checks on the yearly triangle collection.  
Uses: `validate_all`.  
Produces: A printed pass/fail message confirming the yearly triangles are internally consistent.

In [9]:
validate_all(yearly_triangles)

All validations passed.


### Step 10 — Optional: export aggregated databases to CSV

Purpose: Optionally write the quarterly and yearly databases to `data/output/` for downstream use or external review. The export lines are commented out by default.  
Produces: (Commented out) `quarterly_from_monthly_database.csv` and `yearly_from_monthly_database.csv` in `data/output/`. Shape confirmation is always displayed.

In [10]:
# Optional export for a normal local session.
# Uncomment these lines when your data/output folder is writable.
#
# output_dir = project_root / "data" / "output"
# output_dir.mkdir(parents=True, exist_ok=True)
# quarterly_database.to_csv(output_dir / "quarterly_from_monthly_database.csv", index=False)
# yearly_database.to_csv(output_dir / "yearly_from_monthly_database.csv", index=False)

quarterly_database.shape, yearly_database.shape

((7380, 9), (495, 9))